In [23]:
import pandas as pd
import numpy as np

data = {
    'student': ['Aman', 'Priya', 'Rahul', 'Sneha', 'Vikram', 'Anjali', 'Karan', 'Divya', 'Rohan', 'Meera'],
    'subject': ['Math', 'Physics', 'Math', 'Chemistry', 'Physics', 'Math', 'Chemistry', 'Physics', 'Math', 'Chemistry'],
    'score': [78, 85, np.nan, 92, 67, 55, 88, np.nan, 73, 91],
    'attempts': [1, 2, 1, 1, 3, 2, 1, 1, 4, 1],
    'passed': [True, True, False, True, True, False, True, True, True, True]
}
df = pd.DataFrame(data)

In [24]:
df.info()
nulll_count=df.isnull().sum()
missing_percentage=((nulll_count)/len(df)*100)
missing_audit=pd.DataFrame({
    'Missing_count':nulll_count,
    'Missing_percentage':missing_percentage,
    'Datatypes':df.dtypes
}
)
display(missing_audit[missing_audit['Missing_count']>0])

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   student   10 non-null     str    
 1   subject   10 non-null     str    
 2   score     8 non-null      float64
 3   attempts  10 non-null     int64  
 4   passed    10 non-null     bool   
dtypes: bool(1), float64(1), int64(1), str(2)
memory usage: 462.0 bytes


,Missing_count,Missing_percentage,Datatypes
score,2,20.0,float64


In [25]:
marks_mean=df.groupby('subject')['score'].transform('mean')
df['score']=df['score'].fillna(marks_mean)
df['score']

0    78.000000
1    85.000000
2    68.666667
3    92.000000
4    67.000000
5    55.000000
6    88.000000
7    76.000000
8    73.000000
9    91.000000
Name: score, dtype: float64

In [26]:
assert df['score'].notnull().all(), 'Score still contains missing values'
df['score']


0    78.000000
1    85.000000
2    68.666667
3    92.000000
4    67.000000
5    55.000000
6    88.000000
7    76.000000
8    73.000000
9    91.000000
Name: score, dtype: float64

In [27]:
df['efficiency']=(df['score']/df['attempts'])
df['efficiency']

0    78.000000
1    42.500000
2    68.666667
3    92.000000
4    22.333333
5    27.500000
6    88.000000
7    76.000000
8    18.250000
9    91.000000
Name: efficiency, dtype: float64

In [28]:
df.groupby('subject')['student'].count().sort_values(ascending=False)

subject
Math         4
Chemistry    3
Physics      3
Name: student, dtype: int64

In [29]:
# Convert passed values to boolean and count students with a single successful attempt
if df['passed'].dtype == object:
    df['passed'] = df['passed'].replace({'True': True, 'False': False})

df['passed'] = df['passed'].astype(bool)
passed_firsttry = df.loc[df['passed'] & (df['attempts'] == 1), 'student'].count()
print(passed_firsttry)

5


In [30]:
results=df.groupby('subject').agg(
    avg_score=('score' ,'mean'),
    max_attemps=('attempts','max'),
    pass_rate=('passed','mean')
)
print(results)

           avg_score  max_attemps  pass_rate
subject                                     
Chemistry  90.333333            1        1.0
Math       68.666667            4        0.5
Physics    76.000000            3        1.0


In [31]:
teacher_map = pd.DataFrame({
    'subject': ['Math', 'Physics', 'Chemistry'],
    'teacher': ['Mr. Rao', 'Ms. Iyer', 'Mr. Singh']
})

In [32]:
df=df.merge(teacher_map , on='subject',how='left')

In [33]:
df

,student,subject,score,attempts,passed,efficiency,teacher
0,Aman,Math,78.000000,1,True,78.000000,Mr. Rao
1,Priya,Physics,85.000000,2,True,42.500000,Ms. Iyer
2,Rahul,Math,68.666667,1,False,68.666667,Mr. Rao
3,Sneha,Chemistry,92.000000,1,True,92.000000,Mr. Singh
4,Vikram,Physics,67.000000,3,True,22.333333,Ms. Iyer
5,Anjali,Math,55.000000,2,False,27.500000,Mr. Rao
6,Karan,Chemistry,88.000000,1,True,88.000000,Mr. Singh
7,Divya,Physics,76.000000,1,True,76.000000,Ms. Iyer
8,Rohan,Math,73.000000,4,True,18.250000,Mr. Rao
9,Meera,Chemistry,91.000000,1,True,91.000000,Mr. Singh


In [34]:
bins=[-1,60,75,90,100]
labels=['f','c','b','a']
df['grade']=pd.cut(df['score'],bins=bins , labels=labels ,right=True)
df['grade']

0    b
1    b
2    c
3    a
4    c
5    f
6    b
7    b
8    c
9    a
Name: grade, dtype: category
Categories (4, str): ['f' < 'c' < 'b' < 'a']

In [35]:
pivot_result=df.pivot_table(
    values='student',
    index='subject',
    columns='grade',
    aggfunc='count',
    fill_value=0
    
    
)
display(pivot_result)

grade,f,c,b,a
subject,,,,
Chemistry,0,0,1,2
Math,1,2,1,0
Physics,0,1,2,0


In [36]:
df=df.sort_values(by=['subject','score'],ascending=[True , False])
df

,student,subject,score,attempts,passed,efficiency,teacher,grade
3,Sneha,Chemistry,92.000000,1,True,92.000000,Mr. Singh,a
9,Meera,Chemistry,91.000000,1,True,91.000000,Mr. Singh,a
6,Karan,Chemistry,88.000000,1,True,88.000000,Mr. Singh,b
0,Aman,Math,78.000000,1,True,78.000000,Mr. Rao,b
8,Rohan,Math,73.000000,4,True,18.250000,Mr. Rao,c
2,Rahul,Math,68.666667,1,False,68.666667,Mr. Rao,c
5,Anjali,Math,55.000000,2,False,27.500000,Mr. Rao,f
1,Priya,Physics,85.000000,2,True,42.500000,Ms. Iyer,b
7,Divya,Physics,76.000000,1,True,76.000000,Ms. Iyer,b
4,Vikram,Physics,67.000000,3,True,22.333333,Ms. Iyer,c


In [37]:
def remark(row):
    if row['score']<60:
        return 'Needs Impeovement'
    elif row['attempts'] > 2:
        return 'Consistent'
    else:
        return 'Good'
df['remarks']=df.apply(remark , axis=1)
df['remarks']

3                 Good
9                 Good
6                 Good
0                 Good
8           Consistent
2                 Good
5    Needs Impeovement
1                 Good
7                 Good
4           Consistent
Name: remarks, dtype: str